# Notebook 02: Análisis Exploratorio y Preprocesamiento
**Proyecto:** Implementación de IA para detección de deslizamientos
**Estudiante:** Lisa | Universidad EAFIT

Este notebook se encarga de la gestión del dataset, visualización de bandas multiespectrales y preparación de datos para el modelo.

## 1. Carga y Gestión de Datos
Subida del archivo `.zip` y configuración de rutas dinámicas.

In [ ]:
import os
import h5py
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files

# --- 1.1 CARGA DEL ARCHIVO ZIP ---
archivo_zip = '/content/landslide4sense.zip'
if not os.path.exists(archivo_zip):
    print("Por favor, selecciona el archivo 'landslide4sense.zip' en tu computadora:")
    uploaded = files.upload()
else:
    print(f"✓ El archivo {archivo_zip} ya está en el sistema.")

# --- 1.2 DESCOMPRESIÓN ---
print("Descomprimiendo...")
with zipfile.ZipFile(archivo_zip, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset_temp')

# --- 1.3 CONFIGURACIÓN DE RUTAS DINÁMICAS ---
# Buscamos 'TrainData' sin importar la profundidad de carpetas
detector = list(Path('/content/dataset_temp').glob('**/TrainData/img'))

if detector:
    DATA_ROOT = detector[0].parent.parent
    print(f"\n✅ Ruta detectada: {DATA_ROOT}")
    
    img_list = sorted(list((DATA_ROOT / "TrainData/img").glob("*.h5")))
    mask_list = sorted(list((DATA_ROOT / "TrainData/mask").glob("*.h5")))
    print(f"Muestras de entrenamiento: {len(img_list)}")
else:
    raise FileNotFoundError("No se encontró la estructura TrainData/img en el ZIP.")

## 2. Visualización de Bandas
El dataset Landslide4Sense contiene 12 bandas. Vamos a visualizar una composición RGB (Bandas 4, 3, 2).

In [ ]:
def load_h5(path):
    with h5py.File(path, 'r') as f:
        # Retorna el primer dataset encontrado
        return np.array(f[list(f.keys())[0]])

# Seleccionamos una muestra aleatoria
idx = np.random.randint(0, len(img_list))
sample_img = load_h5(img_list[idx])
sample_mask = load_h5(mask_list[idx])

# Composición RGB (Normalizada para visualización)
# Sentinel-2: B4(red)=index 3, B3(green)=index 2, B2(blue)=index 1
rgb = sample_img[:, :, [3, 2, 1]]
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(rgb)
ax[0].set_title(f"Imagen RGB (Sample {idx})")
ax[0].axis('off')

ax[1].imshow(sample_mask, cmap='jet')
ax[1].set_title("Máscara de Deslizamiento")
ax[1].axis('off')

plt.show()

## 3. Estadísticas del Dataset
Análisis de la distribución de clases (píxeles con vs sin deslizamiento).

In [ ]:
positives = 0
n_check = min(100, len(mask_list))

for i in range(n_check):
    m = load_h5(mask_list[i])
    if np.max(m) > 0: positives += 1

print(f"Análisis de {n_check} muestras:")
print(f"Parches con deslizamientos: {positives}")
print(f"Parches sin deslizamientos: {n_check - positives}")